In [ ]:
import os
import ast
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from collections import defaultdict
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({
    "figure.dpi": 130,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA_DIR = "data"

OUT_DIR = "./figures"
os.makedirs(OUT_DIR, exist_ok=True)

CSV_FILES = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])
print(f"Found {len(CSV_FILES)} CSV files in '{DATA_DIR}'")
print("First 5:", CSV_FILES[:5])


## Helper Functions

In [ ]:
def parse_ref_list(val):
    """
    Parse a referenced_works cell which may be:
      - a Python list literal  "["W123", "W456"]"
      - a JSON array           "["W123"]"
      - a plain string         "W123 W456"
      - NaN / None
    Returns a list of uppercase OpenAlex IDs.
    """
    if pd.isna(val) or val == "" or val == "[]":
        return []
    if isinstance(val, list):
        return [str(v).upper() for v in val]
    s = str(val).strip()
    if s.startswith("["):
        try:
            items = ast.literal_eval(s)
            return [str(i).upper() for i in items]
        except Exception:
            pass
    return [v.strip().upper() for v in s.replace(",", " ").split() if v.strip()]


def build_graph(df):
    """
    Build a directed citation graph from a SYNERGY+ CSV.
    Nodes: papers in the dataset (identified by openalex_id).
    Edges: (paper_A → paper_B) if paper_B is in paper_A's referenced_works.
    Node attributes: label (0/1), year, cited_by_count.
    Returns: nx.DiGraph, set of internal node ids (lowercase for matching).
    """
    # Normalise IDs — CSVs use lowercase 'w', referenced_works lists use uppercase 'W'
    df = df.copy()
    df["_id"] = df["openalex_id"].str.upper().str.strip()
    internal_ids = set(df["_id"].dropna())

    G = nx.DiGraph()

    for _, row in df.iterrows():
        nid = row["_id"]
        if pd.isna(nid):
            continue
        G.add_node(
            nid,
            label=int(row.get("label_included", 0)),
            year=row.get("publication_year", None),
            cited_by=row.get("cited_by_count", 0),
            title=str(row.get("title", ""))[:80],
        )

    for _, row in df.iterrows():
        src = row["_id"]
        if pd.isna(src) or src not in internal_ids:
            continue
        refs = parse_ref_list(row.get("referenced_works", ""))
        for ref in refs:
            if ref in internal_ids and ref != src:
                G.add_edge(src, ref)

    return G, internal_ids


def graph_stats(G, df):
    """Return a dict of summary statistics for one dataset."""
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    n_included = df["label_included"].sum()
    inclusion_rate = n_included / len(df) if len(df) else 0
    density = nx.density(G) if n_nodes > 1 else 0
    degrees = [d for _, d in G.degree()]
    avg_degree = np.mean(degrees) if degrees else 0
    max_degree = max(degrees) if degrees else 0
    # giant component
    undirected = G.to_undirected()
    components = list(nx.connected_components(undirected))
    giant_size = max(len(c) for c in components) if components else 0
    giant_frac = giant_size / n_nodes if n_nodes else 0
    # isolated nodes (degree 0)
    isolated = sum(1 for d in degrees if d == 0)
    return {
        "n_nodes": n_nodes,
        "n_edges": n_edges,
        "n_included": int(n_included),
        "inclusion_rate": round(inclusion_rate * 100, 1),
        "density": round(density * 1000, 3),   # × 1000 for readability
        "avg_degree": round(avg_degree, 2),
        "max_degree": int(max_degree),
        "giant_frac": round(giant_frac * 100, 1),
        "n_isolated": int(isolated),
    }

print("✅ Helper functions defined.")


## Load All Datasets

In [ ]:
dataframes = {}
graphs     = {}
stats_rows = []

for fname in tqdm(CSV_FILES, desc="Loading datasets"):
    name = fname.replace(".csv", "")
    try:
        df = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
        # Ensure required columns exist
        for col in ["openalex_id", "label_included"]:
            if col not in df.columns:
                print(f"  ⚠️  {name}: missing column '{col}', skipping")
                continue
        if "referenced_works" not in df.columns:
            df["referenced_works"] = ""
        G, _ = build_graph(df)
        dataframes[name] = df
        graphs[name]     = G
        row = {"dataset": name, **graph_stats(G, df)}
        stats_rows.append(row)
    except Exception as e:
        print(f"  ❌ {name}: {e}")

stats_df = pd.DataFrame(stats_rows).sort_values("n_nodes", ascending=False).reset_index(drop=True)
print(f"\n✅ Loaded {len(dataframes)} datasets successfully.")
print(f"   {len(stats_rows) - len(dataframes)} failed to load.")


## Summary Table

In [ ]:
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:.2f}".format)

display_df = stats_df.copy()
display_df.columns = [
    "Dataset", "Nodes", "Edges", "Included", "Incl %",
    "Density×1k", "Avg Deg", "Max Deg", "Giant %", "Isolated"
]
display(display_df.style
    .background_gradient(subset=["Nodes"], cmap="Blues")
    .background_gradient(subset=["Edges"], cmap="Greens")
    .background_gradient(subset=["Incl %"], cmap="Oranges")
    .background_gradient(subset=["Density×1k"], cmap="Purples")
    .format(precision=1)
)


In [ ]:
stats_df.to_csv(os.path.join(OUT_DIR, "dataset_summary_stats.csv"), index=False)
print("Saved → figures/dataset_summary_stats.csv")


## Cross-Dataset Bar Chart Comparison

In [ ]:
top_n = min(40, len(stats_df))   # show top 40 by size
s = stats_df.head(top_n)

fig, axes = plt.subplots(4, 1, figsize=(max(14, top_n * 0.38), 16))
fig.suptitle("SYNERGY+ / FORAS — Cross-Dataset Comparison", fontsize=14, fontweight="bold", y=0.995)

labels = [d.replace("_", " ") for d in s["dataset"]]
x = np.arange(len(labels))
bar_kw = dict(width=0.72, edgecolor="none")

# — Panel 1: Node count ——————————————————————————
ax = axes[0]
bars = ax.bar(x, s["n_nodes"], color="#378ADD", **bar_kw)
ax.set_title("Number of nodes (papers)", fontsize=11)
ax.set_ylabel("N nodes")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=7.5)

# — Panel 2: Edge count ——————————————————————————
ax = axes[1]
ax.bar(x, s["n_edges"], color="#1D9E75", **bar_kw)
ax.set_title("Number of citation edges", fontsize=11)
ax.set_ylabel("N edges")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=7.5)

# — Panel 3: Inclusion rate ——————————————————————
ax = axes[2]
colors_inc = ["#D85A30" if r > 20 else "#EF9F27" if r > 10 else "#378ADD" for r in s["inclusion_rate"]]
ax.bar(x, s["inclusion_rate"], color=colors_inc, **bar_kw)
ax.axhline(y=10, color="gray", linestyle="--", linewidth=0.8, alpha=0.6, label="10%")
ax.axhline(y=20, color="gray", linestyle=":",  linewidth=0.8, alpha=0.6, label="20%")
ax.set_title("Inclusion rate (%)", fontsize=11)
ax.set_ylabel("% included")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=7.5)
ax.legend(fontsize=8)

# — Panel 4: Graph density × 1000 ———————————————
ax = axes[3]
ax.bar(x, s["density"], color="#7F77DD", **bar_kw)
ax.set_title("Graph density × 1000 (higher = more connected)", fontsize=11)
ax.set_ylabel("Density × 1000")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=7.5)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cross_dataset_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figures/cross_dataset_comparison.png")


## Scatter: Size vs Graph Density (Dataset Selection Map)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

sc = ax.scatter(
    stats_df["n_nodes"],
    stats_df["density"],
    c=stats_df["inclusion_rate"],
    cmap="RdYlGn",
    s=stats_df["n_included"].clip(5, 300) * 3,
    alpha=0.78,
    edgecolors="white",
    linewidths=0.5,
)

cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label("Inclusion rate (%)", fontsize=10)

# Label every point
for _, row in stats_df.iterrows():
    short = row["dataset"].replace("_", " ")
    ax.annotate(
        short, (row["n_nodes"], row["density"]),
        fontsize=6.5, alpha=0.85,
        xytext=(4, 3), textcoords="offset points"
    )

ax.set_xlabel("Number of nodes (papers)", fontsize=11)
ax.set_ylabel("Graph density × 1000", fontsize=11)
ax.set_title(
    "Dataset selection map\n"
    "Bubble size = number of included papers · Colour = inclusion rate",
    fontsize=12, fontweight="bold"
)
ax.set_xscale("log")

# Quadrant lines (medians)
med_nodes   = stats_df["n_nodes"].median()
med_density = stats_df["density"].median()
ax.axvline(med_nodes,   color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(med_density, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.text(med_nodes * 1.05, ax.get_ylim()[1] * 0.97, "median size", fontsize=8, color="gray")
ax.text(ax.get_xlim()[0] * 1.05, med_density * 1.05, "median density", fontsize=8, color="gray")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "dataset_selection_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figures/dataset_selection_scatter.png")


## Citation Network Panels — All Datasets

Nodes coloured: 🔴 included (relevant) | 🔵 excluded (irrelevant)  
Layout: spring (force-directed), seed=42 for reproducibility.  
⚠️ Large datasets (>2000 nodes) are subsampled to 2000 for speed.


In [ ]:
SUBSAMPLE_THRESHOLD = 2000   # nodes above this get subsampled
MAX_SPRING_ITER     = 30     # fewer iterations = faster for large graphs
NODE_SIZE_BASE      = 8

names_sorted = sorted(graphs.keys(), key=lambda k: graphs[k].number_of_nodes(), reverse=True)
n_datasets   = len(names_sorted)
n_cols       = 4
n_rows       = (n_datasets + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.5, n_rows * 4.0))
fig.suptitle("SYNERGY+ Citation Networks · All Datasets", fontsize=14, fontweight="bold")
axes_flat = axes.flatten()

COLOR_INCL = "#E8593C"   # red-orange — included
COLOR_EXCL = "#378ADD"   # blue — excluded

for idx, name in enumerate(names_sorted):
    ax   = axes_flat[idx]
    G    = graphs[name]
    df   = dataframes[name]
    stat = stats_df[stats_df["dataset"] == name].iloc[0]

    # Subsample if too large
    nodes = list(G.nodes())
    if len(nodes) > SUBSAMPLE_THRESHOLD:
        rng   = np.random.default_rng(42)
        nodes = list(rng.choice(nodes, SUBSAMPLE_THRESHOLD, replace=False))
        G_vis = G.subgraph(nodes).copy()
        sampled = True
    else:
        G_vis   = G
        sampled = False

    # Layout
    try:
        pos = nx.spring_layout(G_vis, seed=42, iterations=MAX_SPRING_ITER, k=1.2)
    except Exception:
        pos = nx.circular_layout(G_vis)

    # Node colours
    node_labels  = nx.get_node_attributes(G_vis, "label")
    node_colors  = [COLOR_INCL if node_labels.get(n, 0) == 1 else COLOR_EXCL for n in G_vis.nodes()]
    node_sizes   = [NODE_SIZE_BASE * 2.5 if node_labels.get(n, 0) == 1 else NODE_SIZE_BASE for n in G_vis.nodes()]

    nx.draw_networkx_nodes(G_vis, pos, ax=ax, node_color=node_colors,
                           node_size=node_sizes, alpha=0.85)
    nx.draw_networkx_edges(G_vis, pos, ax=ax,
                           edge_color="#888888", alpha=0.18, width=0.4,
                           arrows=False)
    ax.axis("off")

    short  = name.replace("_", " ")
    sample_note = f"\n(subsample {SUBSAMPLE_THRESHOLD}/{stat['n_nodes']})" if sampled else ""
    title  = (f"{short}{sample_note}\n"
              f"{int(stat['n_nodes'])}n · {int(stat['n_edges'])}e | "
              f"{stat['inclusion_rate']}% incl.")
    ax.set_title(title, fontsize=7.5, pad=3)

# Hide unused axes
for idx in range(n_datasets, len(axes_flat)):
    axes_flat[idx].axis("off")

# Legend
legend_patches = [
    mpatches.Patch(color=COLOR_INCL, label="Included (relevant)"),
    mpatches.Patch(color=COLOR_EXCL, label="Excluded (irrelevant)"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=2,
           fontsize=9, frameon=False, bbox_to_anchor=(0.5, 0.0))

plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig(os.path.join(OUT_DIR, "all_citation_networks.png"), dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → figures/all_citation_networks.png")


## Degree Distribution Plots

Log-log degree distributions. A straight line on log-log = power law (hub structure).  
Hub structure = GNN-friendly: high-degree relevant papers act as anchors.


In [ ]:
names_sorted_deg = sorted(
    graphs.keys(),
    key=lambda k: graphs[k].number_of_nodes(),
    reverse=True
)

n_datasets = len(names_sorted_deg)
n_cols     = 5
n_rows     = (n_datasets + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.2, n_rows * 2.8))
fig.suptitle("Degree Distributions (log–log scale)", fontsize=13, fontweight="bold")
axes_flat = axes.flatten()

for idx, name in enumerate(names_sorted_deg):
    ax = axes_flat[idx]
    G  = graphs[name]
    undirected = G.to_undirected()
    degrees = [d for _, d in undirected.degree()]

    if not degrees or max(degrees) == 0:
        ax.text(0.5, 0.5, "no edges", ha="center", va="center",
                transform=ax.transAxes, fontsize=8, color="gray")
        ax.set_title(name.replace("_", " "), fontsize=7)
        ax.axis("off")
        continue

    # Degree count histogram in log-log
    deg_counts = defaultdict(int)
    for d in degrees:
        deg_counts[d] += 1
    xs = sorted(deg_counts.keys())
    ys = [deg_counts[x] for x in xs]
    xs_pos = [x for x, y in zip(xs, ys) if x > 0 and y > 0]
    ys_pos = [y for x, y in zip(xs, ys) if x > 0 and y > 0]

    if xs_pos:
        ax.scatter(xs_pos, ys_pos, s=8, color="#378ADD", alpha=0.7, linewidths=0)
        ax.set_xscale("log"); ax.set_yscale("log")
    else:
        ax.bar([0], [deg_counts.get(0, 0)], color="#378ADD")

    n_isolated = sum(1 for d in degrees if d == 0)
    iso_pct    = 100 * n_isolated / len(degrees)
    short      = name.replace("_", " ")
    ax.set_title(f"{short}\n{iso_pct:.0f}% isolated", fontsize=7, pad=2)
    ax.set_xlabel("degree", fontsize=7)
    ax.set_ylabel("count",  fontsize=7)
    ax.tick_params(labelsize=6)

for idx in range(n_datasets, len(axes_flat)):
    axes_flat[idx].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "degree_distributions.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Saved → figures/degree_distributions.png")


## Dataset Selection Scorecard

Composite score for thesis suitability. Higher = better candidate.

**Scoring criteria:**
- **Size score** (0–3): ≥1000 nodes = 3, ≥500 = 2, ≥200 = 1, else 0
- **Density score** (0–3): density×1k ≥ 2 = 3, ≥ 1 = 2, ≥ 0.3 = 1, else 0
- **Inclusion score** (0–3): 5–30% = 3 (ideal for imbalanced learning), <5% or >30% = 2 (still useful but harder), else 1
- **Coverage score** (0–1): giant component ≥ 60% of nodes = 1 (graph is connected enough to propagate messages)


In [ ]:
def score_dataset(row):
    # Size
    size_score = 3 if row["n_nodes"] >= 1000 else 2 if row["n_nodes"] >= 500 else 1 if row["n_nodes"] >= 200 else 0
    # Density (density column is already ×1000)
    den = row["density"]
    density_score = 3 if den >= 2 else 2 if den >= 1 else 1 if den >= 0.3 else 0
    # Inclusion rate
    ir = row["inclusion_rate"]
    inclusion_score = 3 if 5 <= ir <= 30 else 2 if (2 <= ir < 5 or 30 < ir <= 50) else 1
    # Giant component coverage
    coverage_score = 1 if row["giant_frac"] >= 60 else 0
    total = size_score + density_score + inclusion_score + coverage_score
    return pd.Series({
        "size_score": size_score,
        "density_score": density_score,
        "inclusion_score": inclusion_score,
        "coverage_score": coverage_score,
        "total_score": total
    })

scores = stats_df.apply(score_dataset, axis=1)
scored_df = pd.concat([stats_df[["dataset", "n_nodes", "n_edges", "inclusion_rate",
                                   "density", "giant_frac"]], scores], axis=1)
scored_df = scored_df.sort_values("total_score", ascending=False).reset_index(drop=True)

# Top 20
top20 = scored_df.head(20)

fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(top20))
w = 0.18
ax.bar(x - 1.5*w, top20["size_score"],      width=w, label="Size (0–3)",      color="#378ADD", alpha=0.85)
ax.bar(x - 0.5*w, top20["density_score"],   width=w, label="Density (0–3)",   color="#1D9E75", alpha=0.85)
ax.bar(x + 0.5*w, top20["inclusion_score"], width=w, label="Inclusion (0–3)", color="#EF9F27", alpha=0.85)
ax.bar(x + 1.5*w, top20["coverage_score"],  width=w, label="Coverage (0–1)",  color="#D85A30", alpha=0.85)

# Total score line
ax2 = ax.twinx()
ax2.plot(x, top20["total_score"], "o--", color="#534AB7", linewidth=1.5, markersize=5, label="Total score")
ax2.set_ylabel("Total score (max 10)", fontsize=10)
ax2.set_ylim(0, 11)

ax.set_xticks(x)
ax.set_xticklabels([d.replace("_", " ") for d in top20["dataset"]], rotation=50, ha="right", fontsize=8)
ax.set_ylabel("Component scores", fontsize=10)
ax.set_title("Dataset Selection Scorecard — Top 20", fontsize=12, fontweight="bold")

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper right")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "dataset_scorecard.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\n🏆 Top 15 datasets by composite score:")
display(scored_df[["dataset", "n_nodes", "n_edges", "inclusion_rate",
                    "density", "giant_frac", "total_score"]].head(15)
        .style.background_gradient(subset=["total_score"], cmap="YlGn").format(precision=1))

scored_df.to_csv(os.path.join(OUT_DIR, "dataset_scorecard.csv"), index=False)
print("Saved → figures/dataset_scorecard.csv")


## Deep Dive — Single Dataset Inspector

Change `INSPECT_DATASET` to any dataset name to get a full breakdown.


In [ ]:
INSPECT_DATASET = "van_de_Schoot_2025"   # <-- change this

if INSPECT_DATASET not in graphs:
    print(f"Dataset '{INSPECT_DATASET}' not found. Available: {list(graphs.keys())[:5]} ...")
else:
    G   = graphs[INSPECT_DATASET]
    df  = dataframes[INSPECT_DATASET]
    und = G.to_undirected()

    print(f"\n{'='*60}")
    print(f"  {INSPECT_DATASET}")
    print(f"{'='*60}")
    print(f"  Nodes (papers)    : {G.number_of_nodes():,}")
    print(f"  Edges (citations) : {G.number_of_edges():,}")
    print(f"  Included papers   : {df['label_included'].sum():,} ({df['label_included'].mean()*100:.1f}%)")
    print(f"  Density           : {nx.density(G):.6f}")

    comps   = sorted(nx.connected_components(und), key=len, reverse=True)
    print(f"  Connected components: {len(comps)}")
    print(f"  Giant component     : {len(comps[0]):,} nodes ({100*len(comps[0])/G.number_of_nodes():.1f}%)")

    degrees = [d for _, d in und.degree()]
    print(f"  Avg degree          : {np.mean(degrees):.2f}")
    print(f"  Max degree          : {max(degrees)}")
    print(f"  Isolated nodes      : {sum(1 for d in degrees if d == 0):,} ({100*sum(1 for d in degrees if d==0)/len(degrees):.1f}%)")

    # Degree of included vs excluded
    incl_nodes = {row["openalex_id"].upper() for _, row in df[df["label_included"]==1].iterrows()}
    deg_incl   = [d for n, d in und.degree() if n in incl_nodes]
    deg_excl   = [d for n, d in und.degree() if n not in incl_nodes]
    print(f"\n  Avg degree (included): {np.mean(deg_incl):.2f}  |  Avg degree (excluded): {np.mean(deg_excl):.2f}")
    print(f"  → {'Included papers are MORE connected ✅' if np.mean(deg_incl) > np.mean(deg_excl) else 'Included papers are LESS connected — GNN may struggle here ⚠️'}")

    # Quick visualisation
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(INSPECT_DATASET.replace("_", " "), fontsize=12, fontweight="bold")

    # — Graph —
    ax = axes[0]
    nodes_vis = list(G.nodes())
    if len(nodes_vis) > 1500:
        rng = np.random.default_rng(42)
        nodes_vis = list(rng.choice(nodes_vis, 1500, replace=False))
    G_vis = G.subgraph(nodes_vis).copy()
    pos   = nx.spring_layout(G_vis, seed=42, iterations=25)
    labels_attr = nx.get_node_attributes(G_vis, "label")
    nc = [COLOR_INCL if labels_attr.get(n, 0) == 1 else COLOR_EXCL for n in G_vis.nodes()]
    ns = [12 if labels_attr.get(n, 0) == 1 else 6 for n in G_vis.nodes()]
    nx.draw_networkx_nodes(G_vis, pos, ax=ax, node_color=nc, node_size=ns, alpha=0.85)
    nx.draw_networkx_edges(G_vis, pos, ax=ax, edge_color="#999", alpha=0.15, width=0.4, arrows=False)
    ax.axis("off"); ax.set_title("Citation network", fontsize=10)

    # — Degree distribution —
    ax = axes[1]
    deg_counts = defaultdict(int)
    for d in degrees: deg_counts[d] += 1
    xs = sorted(deg_counts.keys()); ys = [deg_counts[x] for x in xs]
    xs_p = [x for x, y in zip(xs, ys) if x > 0 and y > 0]
    ys_p = [y for x, y in zip(xs, ys) if x > 0 and y > 0]
    if xs_p:
        ax.scatter(xs_p, ys_p, s=10, color="#378ADD", alpha=0.7)
        ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("degree"); ax.set_ylabel("count")
    ax.set_title("Degree distribution (log-log)", fontsize=10)

    # — Inclusion rate by component size —
    ax = axes[2]
    comp_sizes, comp_incl_rates = [], []
    for comp in comps:
        sub_df = df[df["openalex_id"].str.upper().isin(comp)]
        if len(sub_df) > 0:
            comp_sizes.append(len(comp))
            comp_incl_rates.append(sub_df["label_included"].mean() * 100)
    ax.scatter(comp_sizes, comp_incl_rates, s=15, color="#D85A30", alpha=0.7)
    ax.axhline(df["label_included"].mean() * 100, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("component size"); ax.set_ylabel("inclusion rate (%)")
    ax.set_title("Inclusion rate by component", fontsize=10)
    ax.set_xscale("log")

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"deepdive_{INSPECT_DATASET}.png"), dpi=130, bbox_inches="tight")
    plt.show()


In [ ]:
OUT_DIR_DEEPS = "./figures/inspections"
os.makedirs(OUT_DIR_DEEPS, exist_ok=True)

for paper in scored_df.head(15)["dataset"]:
    INSPECT_DATASET = paper

    if INSPECT_DATASET not in graphs:
        print(f"Dataset '{INSPECT_DATASET}' not found. Available: {list(graphs.keys())[:5]} ...")
    else:
        G   = graphs[INSPECT_DATASET]
        df  = dataframes[INSPECT_DATASET]
        und = G.to_undirected()

        comps   = sorted(nx.connected_components(und), key=len, reverse=True)
        degrees = [d for _, d in und.degree()]

        incl_nodes = {row["openalex_id"].upper() for _, row in df[df["label_included"]==1].iterrows()}
        deg_incl   = [d for n, d in und.degree() if n in incl_nodes]
        deg_excl   = [d for n, d in und.degree() if n not in incl_nodes]

        iso_count = sum(1 for d in degrees if d == 0)
        gnn_signal = "Included MORE connected ✅" if np.mean(deg_incl) > np.mean(deg_excl) else "Included LESS connected ⚠️"

        stats_lines = [
            f"Nodes (papers):    {G.number_of_nodes():,}",
            f"Edges (citations): {G.number_of_edges():,}",
            f"Included:          {df['label_included'].sum():,}",
            f"  ({df['label_included'].mean()*100:.1f}%)",
            f"Density:           {nx.density(G):.6f}",
            "",
            f"Components:        {len(comps)}",
            f"Giant component:   {len(comps[0]):,} nodes",
            f"  ({100*len(comps[0])/G.number_of_nodes():.1f}% of nodes)",
            "",
            f"Avg degree:        {np.mean(degrees):.2f}",
            f"Max degree:        {max(degrees)}",
            f"Isolated nodes:    {iso_count:,}",
            f"  ({100*iso_count/len(degrees):.1f}%)",
            "",
            f"Avg deg (incl):    {np.mean(deg_incl):.2f}",
            f"Avg deg (excl):    {np.mean(deg_excl):.2f}",
            "",
            gnn_signal,
        ]

        print(f"\n{'='*60}")
        print(f"  {INSPECT_DATASET}")
        print(f"{'='*60}")
        for line in stats_lines:
            print(f"  {line}")

        # Visualisation — 4 panels: stats | graph | degree dist | inclusion by component
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(INSPECT_DATASET.replace("_", " "), fontsize=12, fontweight="bold")

        # — Stats text panel —
        ax = axes[0]
        ax.axis("off")
        ax.text(
            0.05, 0.97, "\n".join(stats_lines),
            transform=ax.transAxes,
            fontsize=9, verticalalignment="top", fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#f5f5f5", edgecolor="#cccccc"),
        )
        ax.set_title("Statistics", fontsize=10)

        # — Citation network —
        ax = axes[1]
        nodes_vis = list(G.nodes())
        if len(nodes_vis) > 1500:
            rng = np.random.default_rng(42)
            nodes_vis = list(rng.choice(nodes_vis, 1500, replace=False))
        G_vis = G.subgraph(nodes_vis).copy()
        pos   = nx.spring_layout(G_vis, seed=42, iterations=25)
        labels_attr = nx.get_node_attributes(G_vis, "label")
        nc = [COLOR_INCL if labels_attr.get(n, 0) == 1 else COLOR_EXCL for n in G_vis.nodes()]
        ns = [12 if labels_attr.get(n, 0) == 1 else 6 for n in G_vis.nodes()]
        nx.draw_networkx_nodes(G_vis, pos, ax=ax, node_color=nc, node_size=ns, alpha=0.85)
        nx.draw_networkx_edges(G_vis, pos, ax=ax, edge_color="#999", alpha=0.15, width=0.4, arrows=False)
        ax.axis("off"); ax.set_title("Citation network", fontsize=10)

        # — Degree distribution —
        ax = axes[2]
        deg_counts = defaultdict(int)
        for d in degrees: deg_counts[d] += 1
        xs = sorted(deg_counts.keys()); ys = [deg_counts[x] for x in xs]
        xs_p = [x for x, y in zip(xs, ys) if x > 0 and y > 0]
        ys_p = [y for x, y in zip(xs, ys) if x > 0 and y > 0]
        if xs_p:
            ax.scatter(xs_p, ys_p, s=10, color="#378ADD", alpha=0.7)
            ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("degree"); ax.set_ylabel("count")
        ax.set_title("Degree distribution (log-log)", fontsize=10)

        # — Inclusion rate by component size —
        ax = axes[3]
        comp_sizes, comp_incl_rates = [], []
        for comp in comps:
            sub_df = df[df["openalex_id"].str.upper().isin(comp)]
            if len(sub_df) > 0:
                comp_sizes.append(len(comp))
                comp_incl_rates.append(sub_df["label_included"].mean() * 100)
        ax.scatter(comp_sizes, comp_incl_rates, s=15, color="#D85A30", alpha=0.7)
        ax.axhline(df["label_included"].mean() * 100, color="gray", linestyle="--", linewidth=0.8)
        ax.set_xlabel("component size"); ax.set_ylabel("inclusion rate (%)")
        ax.set_title("Inclusion rate by component", fontsize=10)
        ax.set_xscale("log")

        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR_DEEPS, f"deepdive_{INSPECT_DATASET}.png"), dpi=130, bbox_inches="tight")
        plt.show()


## Connectivity vs Relevance — Across All Datasets

Key question for your thesis: **are included papers more connected than excluded ones?**  
If yes, graph structure carries signal. If no, text features dominate.


In [ ]:
connectivity_rows = []

for name in graphs:
    G   = graphs[name]
    df  = dataframes[name]
    und = G.to_undirected()
    deg_map = dict(und.degree())

    for _, row in df.iterrows():
        nid = str(row.get("openalex_id", "")).upper().strip()
        d   = deg_map.get(nid, 0)
        connectivity_rows.append({
            "dataset": name,
            "label":   int(row.get("label_included", 0)),
            "degree":  d,
        })

conn_df = pd.DataFrame(connectivity_rows)

# Average degree per label per dataset
summary = conn_df.groupby(["dataset", "label"])["degree"].mean().unstack(fill_value=0)
summary.columns = ["excl_avg_deg", "incl_avg_deg"]
summary["ratio"] = (summary["incl_avg_deg"] / summary["excl_avg_deg"].replace(0, np.nan)).fillna(1)
summary = summary.sort_values("ratio", ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(summary))
ax.bar(x - 0.2, summary["incl_avg_deg"], width=0.38, color=COLOR_INCL, alpha=0.85, label="Included papers")
ax.bar(x + 0.2, summary["excl_avg_deg"], width=0.38, color=COLOR_EXCL, alpha=0.85, label="Excluded papers")
ax.set_xticks(x)
ax.set_xticklabels([d.replace("_", " ") for d in summary.index],
                   rotation=55, ha="right", fontsize=7.5)
ax.set_ylabel("Average node degree")
ax.set_title(
    "Average degree: included vs excluded papers per dataset\n"
    "Datasets where red > blue → graph structure has signal for GNN",
    fontsize=11, fontweight="bold"
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "inclusion_vs_degree.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\n📊 Datasets where INCLUDED papers are more connected (good for GNN):")
gnn_friendly = summary[summary["ratio"] > 1.2].sort_values("ratio", ascending=False)
print(gnn_friendly[["incl_avg_deg", "excl_avg_deg", "ratio"]].to_string())


## Export — Final Summary for Dataset Selection Meeting

In [ ]:
final = scored_df.merge(
    summary.reset_index().rename(columns={"dataset": "dataset"}),
    on="dataset", how="left"
)
final.to_csv(os.path.join(OUT_DIR, "final_dataset_analysis.csv"), index=False)

print("✅ All outputs saved to ./figures/")
print("\nFiles generated:")
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f"  {f:50s} {size/1024:.1f} KB")
